In [30]:
import pandas as pd
import plotly.express as px

In [31]:
df = pd.read_csv(
    r"C:\Users\jidny\OneDrive\Documents\Hiresnix_Internship\Project_1_Accident_Severity_Predictor\Dataset\Road.csv"
)

df.head()

,Time,Day_of_week,Age_band_of_driver,Sex_of_driver,Educational_level,Vehicle_driver_relation,Driving_experience,Type_of_vehicle,Owner_of_vehicle,Service_year_of_vehicle,...,Vehicle_movement,Casualty_class,Sex_of_casualty,Age_band_of_casualty,Casualty_severity,Work_of_casuality,Fitness_of_casuality,Pedestrian_movement,Cause_of_accident,Accident_severity
0,17:02:00,Monday,18-30,Male,Above high school,Employee,1-2yr,Automobile,Owner,Above 10yr,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Moving Backward,Slight Injury
1,17:02:00,Monday,31-50,Male,Junior high school,Employee,Above 10yr,Public (> 45 seats),Owner,5-10yrs,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Overtaking,Slight Injury
2,17:02:00,Monday,18-30,Male,Junior high school,Employee,1-2yr,Lorry (41?100Q),Owner,NaN,...,Going straight,Driver or rider,Male,31-50,3,Driver,NaN,Not a Pedestrian,Changing lane to the left,Serious Injury
3,1:06:00,Sunday,18-30,Male,Junior high school,Employee,5-10yr,Public (> 45 seats),Governmental,NaN,...,Going straight,Pedestrian,Female,18-30,3,Driver,Normal,Not a Pedestrian,Changing lane to the right,Slight Injury
4,1:06:00,Sunday,18-30,Male,Junior high school,Employee,2-5yr,NaN,Owner,5-10yrs,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Overtaking,Slight Injury


In [32]:
df["Time"].head(10)

0    17:02:00
1    17:02:00
2    17:02:00
3     1:06:00
4     1:06:00
5    14:15:00
6    17:30:00
7    17:20:00
8    17:20:00
9    17:20:00
Name: Time, dtype: object

In [33]:
df["Time"].dtype

dtype('O')

In [34]:
df["Time_Converted"] = pd.to_datetime(
    df["Time"],
    errors="coerce"
)

C:\Users\jidny\AppData\Local\Temp\ipykernel_21168\961376386.py:1: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [35]:
df[
    ["Time", "Time_Converted"]
].head()

,Time,Time_Converted
0,17:02:00,2026-07-31 17:02:00
1,17:02:00,2026-07-31 17:02:00
2,17:02:00,2026-07-31 17:02:00
3,1:06:00,2026-07-31 01:06:00
4,1:06:00,2026-07-31 01:06:00


In [36]:
df["Accident_Hour"] = (
    df["Time_Converted"]
    .dt.hour
)

df[
    ["Time", "Accident_Hour"]
].head()

,Time,Accident_Hour
0,17:02:00,17
1,17:02:00,17
2,17:02:00,17
3,1:06:00,1
4,1:06:00,1


In [37]:
def get_time_period(hour):

    if 5 <= hour < 12:
        return "Morning"

    elif 12 <= hour < 17:
        return "Afternoon"

    elif 17 <= hour < 21:
        return "Evening"

    else:
        return "Night"

In [38]:
df["Time_Period"] = (
    df["Accident_Hour"]
    .apply(get_time_period)
)

df[
    ["Time", "Accident_Hour", "Time_Period"]
].head()

,Time,Accident_Hour,Time_Period
0,17:02:00,17,Evening
1,17:02:00,17,Evening
2,17:02:00,17,Evening
3,1:06:00,1,Night
4,1:06:00,1,Night


In [39]:
time_period_counts = (
    df["Time_Period"]
    .value_counts()
    .reindex(
        [
            "Morning",
            "Afternoon",
            "Evening",
            "Night"
        ]
    )
    .reset_index()
)

time_period_counts.columns = [
    "Time_Period",
    "Accident_Count"
]

time_period_counts

,Time_Period,Accident_Count
0,Morning,3312
1,Afternoon,3897
2,Evening,3496
3,Night,1611


In [40]:
fig = px.bar(
    time_period_counts,
    x="Time_Period",
    y="Accident_Count",
    title="Accident Frequency by Time of Day",
    text="Accident_Count"
)

fig.update_layout(
    xaxis_title="Time of Day",
    yaxis_title="Number of Accidents"
)

fig.show()

In [41]:
time_severity = pd.crosstab(
    df["Time_Period"],
    df["Accident_severity"]
)

time_severity = time_severity.reindex(
    [
        "Morning",
        "Afternoon",
        "Evening",
        "Night"
    ]
)

time_severity

Accident_severity,Fatal injury,Serious Injury,Slight Injury
Time_Period,,,
Morning,25,447,2840
Afternoon,39,535,3323
Evening,48,492,2956
Night,46,269,1296


In [42]:
time_severity_plot = (
    time_severity
    .reset_index()
)

fig = px.bar(
    time_severity_plot,
    x="Time_Period",
    y=[
        "Fatal injury",
        "Serious Injury",
        "Slight Injury"
    ],
    title="Accident Severity by Time of Day",
    barmode="group"
)

fig.update_layout(
    xaxis_title="Time Period",
    yaxis_title="Number of Accidents"
)

fig.show()

In [43]:
fig = px.imshow(
    time_severity,
    text_auto=True,
    aspect="auto",
    title="Accident Severity Heatmap by Time of Day",
    labels={
        "x": "Accident Severity",
        "y": "Time Period",
        "color": "Number of Accidents"
    }
)

fig.show()

In [44]:
hourly_counts = (
    df["Accident_Hour"]
    .value_counts()
    .sort_index()
    .reset_index()
)

hourly_counts.columns = [
    "Hour",
    "Accident_Count"
]

hourly_counts

,Hour,Accident_Count
0,0,206
1,1,134
2,2,84
3,3,84
4,4,91
5,5,76
6,6,214
7,7,532
8,8,828
9,9,559


In [45]:
fig = px.line(
    hourly_counts,
    x="Hour",
    y="Accident_Count",
    markers=True,
    title="Accident Frequency by Hour"
)

fig.update_layout(
    xaxis_title="Hour of the Day",
    yaxis_title="Number of Accidents",
    xaxis=dict(
        tickmode="linear",
        dtick=1
    )
)

fig.show()

In [46]:
time_period_counts.to_csv(
    r"C:\Users\jidny\OneDrive\Documents\Hiresnix_Internship\Project_1_Accident_Severity_Predictor\Report\day28_time_period_counts.csv",
    index=False
)

time_severity.to_csv(
    r"C:\Users\jidny\OneDrive\Documents\Hiresnix_Internship\Project_1_Accident_Severity_Predictor\Report\day28_time_severity.csv"
)

hourly_counts.to_csv(
    r"C:\Users\jidny\OneDrive\Documents\Hiresnix_Internship\Project_1_Accident_Severity_Predictor\Report\day28_hourly_accident_counts.csv",
    index=False
)

print("Day 28 analysis files saved successfully!")

Day 28 analysis files saved successfully!
